# 5.1 코드 에이전트 개요


코드 에이전트(Code Agent)란:
- **정의**: LLM이 코드를 생성하고 수정하고 최적화하는 자율 에이전트
- **특징**: 코드 → 실행 → 피드백 → 개선 의 반복 루프
- **목표**: 사람의 개입 없이 프로그래밍 문제를 자동으로 해결한다

**실제 사용 사례**:
- GitHub Copilot: 코드 자동완성 에이전트
- Devin AI: 소프트웨어 엔지니어링 에이전트
- AutoGPT: 자동화 작업 에이전트

**성능**:
- 초기 세대(2023): 약 5-10% 문제 해결률
- 최신 세대(2024): 약 30-50% 문제 해결률
- 규모가 클수록, 더 많은 반복 기회 주면 성능 향상한다

## 코드 에이전트의 RL 관점 분석

### MDP로서의 코드 에이전트

```
상태 (State):
  - 현재 코드
  - 실행 오류 메시지
  - 테스트 케이스 결과
  → 상태 = (code, errors, test_results)

행동 (Action):
  - 코드 생성
  - 코드 수정
  - 디버깅
  → 행동 = 텍스트 (LLM이 선택)

보상 (Reward):
  - 테스트 통과: +1 per test
  - 코드 품질: +0.5 per quality criterion
  - 오류 감소: +1 per error fixed
  → 최종 보상 = 테스트통과율 * 100

전이 확률:
  - 코드 → exec() → 오류/테스트 결과
  - 결정적: 같은 코드는 같은 결과
```

### Generative Agent Memory와의 매핑

**Generative Agent Memory** (3.3 Module 참고):
```
Perceive → Memory Stream → Retrieve → Act
```

**Code Agent로의 매핑**:
```
Perceive: 코드 실행 결과 수집한다
  - 오류 메시지
  - 테스트 실패
  - 런타임 정보

Memory Stream: 이전 오류와 해결 방법 저장한다
  - "IndexError 발생 시 범위 확인한다"
  - "TypeError 발생 시 타입 변환 필요하다"
  - "테스트 케이스 예시들"

Retrieve: 현재 오류 패턴과 유사한 과거 경험 검색한다
  - 같은 타입의 오류? → 과거 해결법 참고한다
  - 비슷한 코드 구조? → 유사 패턴 참고한다

Act: 검색한 경험을 바탕으로 코드 개선한다
  - LLM이 수정 코드 생성한다
  - 메모리 기반 가이드로 더 나은 수정한다
```

In [1]:
from utils_openai import *
import numpy as np
import re

heading("준비: 코드 에이전트")
print("✓ ask() - LLM으로 코드 생성한다")
print("✓ exec() - Python 코드 실행한다")
print("✓ llm_reward() - 코드 품질 평가한다")
print("✓ MemoryStream - 오류 패턴 학습한다")

def safe_llm_reward(text, criteria="정확성, 완전성, 유용성", max_score=10.0):
    """llm_reward의 안전한 래퍼이다. 숫자 파싱을 보장한다."""
    score_str = ask(
        f"평가 기준: {criteria}\n응답: {text}\n\n0~{max_score} 사이 숫자 하나만 반환하라. 설명 없이 숫자만 출력하라. 예: 7.5",
        temperature=0.0, max_tokens=5
    )
    numbers = re.findall(r'[\d.]+', score_str)
    return min(max(float(numbers[0]), 0.0), max_score) if numbers else max_score * 0.5

def strip_code_blocks(code_text):
    """LLM 응답에서 마크다운 코드 블록을 제거한다."""
    if "```" in code_text:
        code_text = code_text.replace("```python", "").replace("```", "").strip()
    return code_text


────────────────────────────────────────
  준비: 코드 에이전트
────────────────────────────────────────
✓ ask() - LLM으로 코드 생성한다
✓ exec() - Python 코드 실행한다
✓ llm_reward() - 코드 품질 평가한다
✓ MemoryStream - 오류 패턴 학습한다


## 실습 1: 간단한 코드 생성 에이전트

LLM에게 파이썬 함수를 작성하도록 하고, 실행해본다.

In [2]:
heading("실습 1: 코드 생성")

problem = "두 정수를 입력받아 최대공약수를 반환하는 파이썬 함수를 작성하라. 함수명은 gcd이다."
step_print(1, "문제", problem)

step_print(2, "코드 생성", "ask()로 LLM 호출한다")

code = ask(
    problem,
    system="파이썬 프로그래머이다. 설명 없이 오직 함수 정의 코드만 반환하라.",
    temperature=0.3
)
code = strip_code_blocks(code)

print(f"생성된 코드:\n{code}")


────────────────────────────────────────
  실습 1: 코드 생성
────────────────────────────────────────
  [Step 1] 문제
    두 정수를 입력받아 최대공약수를 반환하는 파이썬 함수를 작성하라. 함수명은 gcd이다.
  [Step 2] 코드 생성
    ask()로 LLM 호출한다
생성된 코드:
def gcd(a, b):
    while b:
        a, b = b, a % b
    return a


## 실습 2: 코드 실행 및 피드백 루프

생성된 코드를 실행하고 오류를 분석한다.

In [3]:
heading("실습 2: 코드 실행")

step_print(1, "코드 실행", "exec()로 함수 정의한다")

namespace = {}
gcd_func = None

exec(code, namespace)
gcd_func = namespace.get('gcd')

if gcd_func:
    print("  함수 정의 성공한다")
else:
    print("  오류: 함수 'gcd'를 찾을 수 없다")

step_print(2, "테스트", "함수 호출 테스트한다")

test_cases = [(12, 8), (100, 50), (17, 13)]
test_results = []

if gcd_func:
    for a, b in test_cases:
        result = gcd_func(a, b)
        expected = np.gcd(a, b)
        passed = result == expected
        test_results.append(passed)
        status = "통과" if passed else "실패"
        print(f"  gcd({a}, {b}) = {result} (기대값: {expected}) [{status}]")

    print()
    step_print(3, "결과", f"테스트 통과율: {sum(test_results)}/{len(test_cases)}")
else:
    print("  함수 정의 실패로 테스트 불가능하다")


────────────────────────────────────────
  실습 2: 코드 실행
────────────────────────────────────────
  [Step 1] 코드 실행
    exec()로 함수 정의한다
  함수 정의 성공한다
  [Step 2] 테스트
    함수 호출 테스트한다
  gcd(12, 8) = 4 (기대값: 4) [통과]
  gcd(100, 50) = 50 (기대값: 50) [통과]
  gcd(17, 13) = 1 (기대값: 1) [통과]

  [Step 3] 결과
    테스트 통과율: 3/3


## 실습 3: 코드 에이전트의 3가지 핵심 능력

코드 에이전트가 갖춰야 할 핵심 능력들을 실습한다.

In [4]:
heading("실습 3: 코드 에이전트의 3가지 핵심 능력")

step_print(1, "능력 1: 생성(Generation)", "처음부터 새로운 코드 작성한다")

problem1 = "두 리스트의 교집합을 반환하는 함수 intersection을 작성하라. 중복 제거한다."
code1 = ask(problem1, system="설명 없이 간단하고 정확한 파이썬 함수만 작성한다.", temperature=0.3)
code1 = strip_code_blocks(code1)
print(f"  생성 완료: {len(code1)} 글자")
print(f"  코드:\n{code1[:200]}")

step_print(2, "능력 2: 반복 수정(Iterative Refinement)", "오류 기반 자동 수정한다")

flawed_code = """def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)
"""

print(f"  원본 코드: {flawed_code.strip()[:60]}...")
print(f"  (재귀 깊이 초과 위험하다)")

refinement_prompt = f"""다음 함수는 매우 느리다 (지수 시간복잡도). 더 효율적으로 수정하라 (O(n) 또는 O(log n)):
{flawed_code}
더 나은 버전의 함수 코드만 반환하라."""

refined = ask(refinement_prompt, system="효율적인 파이썬 함수를 작성한다. 코드만 반환한다.", temperature=0.3)
refined = strip_code_blocks(refined)
print(f"  수정 완료:\n{refined[:200]}")

step_print(3, "능력 3: 자동화(Automation)", "다단계 작업 자동 수행한다")

print("  시뮬레이션: 데이터 처리 파이프라인 자동 생성한다")
print("    1단계: 입력 검증 함수 자동 생성한다")
print("    2단계: 데이터 정제 함수 자동 생성한다")
print("    3단계: 출력 포맷팅 함수 자동 생성한다")
print("    4단계: 모든 함수를 파이프라인으로 연결한다")
print("  → 결과: 완전 자동화된 데이터 처리 시스템")


────────────────────────────────────────
  실습 3: 코드 에이전트의 3가지 핵심 능력
────────────────────────────────────────
  [Step 1] 능력 1: 생성(Generation)
    처음부터 새로운 코드 작성한다
  생성 완료: 72 글자
  코드:
def intersection(list1, list2):
    return list(set(list1) & set(list2))
  [Step 2] 능력 2: 반복 수정(Iterative Refinement)
    오류 기반 자동 수정한다
  원본 코드: def fibonacci(n):
    if n <= 1:
        return n
    return...
  (재귀 깊이 초과 위험하다)
  수정 완료:
def fibonacci(n):
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b
  [Step 3] 능력 3: 자동화(Automation)
    다단계 작업 자동 수행한다
  시뮬레이션: 데이터 처리 파이프라인 자동 생성한다
    1단계: 입력 검증 함수 자동 생성한다
    2단계: 데이터 정제 함수 자동 생성한다
    3단계: 출력 포맷팅 함수 자동 생성한다
    4단계: 모든 함수를 파이프라인으로 연결한다
  → 결과: 완전 자동화된 데이터 처리 시스템


## 실습 4: LLM-as-Judge를 사용한 코드 품질 평가

생성된 코드의 품질을 여러 기준으로 평가한다.

In [5]:
heading("실습 4: 코드 품질 평가")

sample_code = """def find_max(numbers):
    if not numbers:
        return None
    max_val = numbers[0]
    for num in numbers[1:]:
        if num > max_val:
            max_val = num
    return max_val
"""

step_print(1, "평가 대상", "코드 샘플")
print(f"{sample_code}")

step_print(2, "다중 기준 평가", "4개 기준으로 점수 계산한다")

evaluation_criteria = {
    "정확성": "코드 논리의 정확성과 버그 여부",
    "가독성": "변수명, 주석, 코드 스타일",
    "효율성": "시간 복잡도와 공간 복잡도",
    "안정성": "엣지 케이스 처리와 예외 처리"
}

evaluation_scores = {}

for criterion_name, criterion_desc in evaluation_criteria.items():
    score = safe_llm_reward(sample_code, criteria=criterion_desc, max_score=10.0)
    evaluation_scores[criterion_name] = score
    print(f"  {criterion_name:8s}: {score:.1f}/10")

print()
step_print(3, "종합 점수", "평균값")
average_score = np.mean(list(evaluation_scores.values()))
print(f"  평균: {average_score:.1f}/10")
print(f"  평가: {'우수하다' if average_score >= 8 else '양호하다' if average_score >= 6 else '개선 필요하다'}")


────────────────────────────────────────
  실습 4: 코드 품질 평가
────────────────────────────────────────
  [Step 1] 평가 대상
    코드 샘플
def find_max(numbers):
    if not numbers:
        return None
    max_val = numbers[0]
    for num in numbers[1:]:
        if num > max_val:
            max_val = num
    return max_val

  [Step 2] 다중 기준 평가
    4개 기준으로 점수 계산한다
  정확성     : 8.3/10
  가독성     : 8.3/10
  효율성     : 8.3/10
  안정성     : 8.3/10

  [Step 3] 종합 점수
    평균값
  평균: 8.3/10
  평가: 우수하다


## 코드 에이전트의 구조

### 전체 루프

```
입력: 프로그래밍 문제
    ↓
1. 코드 생성 (ask + LLM)
    ↓
2. 코드 실행 (exec + test)
    ↓
  성공? → 반환
    ↓
  실패? → 오류 분석
    ↓
3. 수정 (ask + error)
    ↓
  반복 (2번으로 돌아간다)
```

### 핵심 특징

1. **폐쇄 루프 제어**
   - 결과 기반 자동 피드백한다
   - 성공까지 반복한다

2. **신호 설계**
   - 오류 메시지 = 부정적 신호
   - 테스트 통과 = 긍정적 신호

3. **RL 관점**
   - Policy: 어떤 코드를 생성할지
   - Reward: 테스트 통과율
   - Value Function: 특정 오류가 얼마나 심각한지 (메모리 통해 학습한다)

## 실습 5: 코드 에이전트의 실제 작동 시뮬레이션

전체 루프를 한 번에 시뮬레이션한다.

In [6]:
heading("실습 5: 전체 에이전트 루프")

problem = "파이썬으로 소수 판별 함수 is_prime을 작성하라. 입력: 정수, 출력: bool"

step_print(1, "문제", problem)

test_cases = [
    (2, True),
    (4, False),
    (17, True),
    (1, False),
    (100, False)
]

step_print(2, "테스트 케이스", "5개")
for num, expected in test_cases:
    print(f"  is_prime({num}) = {expected}")

step_print(3, "루프 시뮬레이션", "반복 시도한다")

print("\n  [라운드 1] 코드 생성한다")
round1_code = ask(problem, system="설명 없이 파이썬 함수만 작성한다.", temperature=0.7)
round1_code = strip_code_blocks(round1_code)
print(f"    생성: {len(round1_code)} 글자")
print(f"    코드 일부: {round1_code[:100]}...")

print("\n  [라운드 2] 피드백 기반 수정한다")
round2_code = ask(
    f"다음 소수 판별 함수를 검토하고 개선하라. 엣지 케이스(0, 1, 음수)를 처리하라:\n{round1_code}",
    system="설명 없이 파이썬 함수만 작성한다.",
    temperature=0.3
)
round2_code = strip_code_blocks(round2_code)
print(f"    수정 완료: {len(round2_code)} 글자")

print("\n  [라운드 3] 최종 검증한다")
namespace = {}
exec(round2_code, namespace)
is_prime_func = namespace.get('is_prime')

passed = 0
if is_prime_func:
    for num, expected in test_cases:
        result = is_prime_func(num)
        if result == expected:
            passed += 1
            print(f"    is_prime({num}) = {result} ✓")
        else:
            print(f"    is_prime({num}) = {result}, 기대: {expected} ✗")

step_print(4, "최종 결과", f"테스트 통과: {passed}/{len(test_cases)}")
print(f"  소요 라운드: 3")


────────────────────────────────────────
  실습 5: 전체 에이전트 루프
────────────────────────────────────────
  [Step 1] 문제
    파이썬으로 소수 판별 함수 is_prime을 작성하라. 입력: 정수, 출력: bool
  [Step 2] 테스트 케이스
    5개
  is_prime(2) = True
  is_prime(4) = False
  is_prime(17) = True
  is_prime(1) = False
  is_prime(100) = False
  [Step 3] 루프 시뮬레이션
    반복 시도한다

  [라운드 1] 코드 생성한다
    생성: 156 글자
    코드 일부: def is_prime(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
       ...

  [라운드 2] 피드백 기반 수정한다
    수정 완료: 156 글자

  [라운드 3] 최종 검증한다
    is_prime(2) = True ✓
    is_prime(4) = False ✓
    is_prime(17) = True ✓
    is_prime(1) = False ✓
    is_prime(100) = False ✓
  [Step 4] 최종 결과
    테스트 통과: 5/5
  소요 라운드: 3


## 요약: 코드 에이전트

### 정의와 특징

**코드 에이전트**는 다음을 갖춘 자율 시스템이다:

1. **생성 능력**: LLM으로 코드 생성한다
2. **실행 능력**: 코드를 실행하고 결과 평가한다
3. **학습 능력**: 오류 피드백으로 자동 개선한다

### RL 관점 정리

- **상태**: 현재 코드 + 오류 메시지
- **행동**: LLM이 생성하는 새 코드
- **보상**: 테스트 통과율 (0~1)
- **정책**: 주어진 상태에서 어떤 코드를 생성할지
- **메모리**: 과거 오류 패턴과 해결 방법

### 성능 향상 방법

1. **더 나은 기본 모델**: GPT-4 vs GPT-3.5 (약 3배 성능 차이)
2. **더 많은 반복**: 3회 vs 10회 반복 (누적 성공률 증가한다)
3. **더 나은 신호**: 테스트 케이스 추가, 상세한 오류 메시지
4. **메모리 활용**: 과거 오류와 해결법 재사용한다